# `passage-eval` — Walkthrough 01: classical A\* through the `Planner` ABC

This is the **canonical illustration** of the `passage-eval` cross-family
abstraction: take a planner that does not look anything like the
released vision baselines (here: a hand-rolled A\* on the analytic
cost grid), implement the `Planner` ABC, score it through
`score_planner`, and emit one run bundle to the leaderboard.

> Paper is frozen (NeurIPS 2026 ED, 2026-05-07). The leaderboard is
> *additive infrastructure*: anything submitted here lands under
> `passage-eval/results/runs/<run_id>/` and never edits the paper-cited
> numbers.

## 1. Imports + version sanity check

In [1]:
import sys, json, socket, getpass, os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PE_SRC = Path('~/projects/passage/passage/src/passage-eval/src')
if str(PE_SRC) not in sys.path: sys.path.insert(0, str(PE_SRC))
import passage_eval as pe
print('passage-eval version:', pe.__version__)


passage-eval version: 0.3.0


## 2. Build a small synthetic batch of `Scene`s

`Scene` is the only thing the protocol asks for: an elevation grid,
an obstacle mask, start + goal cells, the cost-model name and weight,
plus the oracle path + its cost. We bootstrap the oracle by running
`AStarPlanner` once with the analytic cost-weight, then freeze its
output as the ground-truth oracle for the rest of the batch.

In [2]:
rng = np.random.default_rng(0)
N, H, W = 8, 32, 32
oracle_planner = pe.AStarPlanner()

scenes = []
for i in range(N):
    elev = rng.uniform(0, 100, size=(H, W)).astype(np.float64)
    obs = (rng.uniform(0, 1, size=(H, W)) < 0.05).astype(np.uint8)
    bootstrap = oracle_planner.predict_path(pe.Scene(
        sample_id=f's{i}', elevation=elev, obstacles=obs,
        start=(0, 0), goal=(H-1, W-1),
        cost_model='elevation', cost_weight=1.0,
        oracle_path=[(0,0)], oracle_cost=0.0,
    ))
    scenes.append(pe.Scene(
        sample_id=f's{i}', elevation=elev, obstacles=obs,
        start=(0, 0), goal=(H-1, W-1),
        cost_model='elevation', cost_weight=1.0,
        oracle_path=bootstrap.predicted_path, oracle_cost=bootstrap.predicted_cost,
    ))
print(f'built {len(scenes)} scenes (resolution={H}x{W}, cost_model=elevation)')


built 8 scenes (resolution=32x32, cost_model=elevation)


## 3. Score two planners through the same `score_planner` driver

`AStarPlanner` (the oracle replicated) and `GreedyToGoalPlanner` (a
weak baseline that ignores costs) both implement `Planner` — the
scoring code is identical for both.

In [3]:
astar_metrics = pe.score_planner(pe.AStarPlanner(),  scenes)
greedy_metrics = pe.score_planner(pe.GreedyToGoalPlanner(), scenes)

import pandas as pd
pd.DataFrame([{'method':'AStarPlanner', **astar_metrics},
              {'method':'GreedyToGoalPlanner', **greedy_metrics}])  [['method','cost_regret_median','cost_regret_p90','path_assd','path_hausdorff_p95','success_rate','n_samples']]


,method,cost_regret_median,cost_regret_p90,path_assd,path_hausdorff_p95,success_rate,n_samples
0,AStarPlanner,0.000000,0.000000,0.000000,0.000000,1.0,8
1,GreedyToGoalPlanner,0.087332,0.123439,0.028163,0.054626,1.0,8


## 4. Build the `Submission` records + write to `results/`

Every leaderboard row carries:

- **identity** — method_name, family, architecture, decoder_variant
- **benchmark cell** — resolution, cost_model, split, seed
- **deployment** — host, edge_accelerator, batch_size, dtype, latency
- **provenance** — package_version, dataset_version, commit_sha, timestamp_utc

The on-disk artefacts go to `results/runs/<run_id>/` (the run_id is
sortable by ISO timestamp + slug).

In [4]:
RESULTS_ROOT = Path('~/projects/passage/passage/src/passage-eval/results')

def submit_classical(name, family, decoder, metrics, latency_ms=None):
    sub = pe.Submission(
        method_name=name,
        family=family,
        architecture='analytic',
        decoder_variant=decoder,
        resolution=H, cost_model='elevation', cost_weight=1.0,
        split='val', seed=1234, n_samples=N,
        metrics=metrics,
        edge_accelerator=None,
        batch_size=1, dtype='fp64',
        latency_e2e_ms_p50=latency_ms,
        package_version=pe.__version__,
        dataset_version='synthetic-32x32-rng0',
        extra={'notebook': '01_astar_walkthrough'},
    )
    sidecar = pe.ReproducibilitySidecar(
        method_name=name, family=family,
        dataset_version='synthetic-32x32-rng0',
        dataset_content_hash='sha256:placeholder',
        operating_point=f'{H}x{W}/elevation',
        seeds=(1234,), batch_size=1, dtype='fp32',
        determinism_flag='algorithmically_deterministic',
        commit_sha='passage-eval-' + pe.__version__,
        notes='classical reference, illustrative cross-family',
    )
    run_dir = pe.submit(sub, root=RESULTS_ROOT, sidecar=sidecar)
    print(f'wrote {run_dir.relative_to(RESULTS_ROOT)}')

submit_classical('astar_classical',  'classical', 'analytic_a_star', astar_metrics, latency_ms=0.5)
submit_classical('greedy_classical', 'classical', 'manhattan_descent', greedy_metrics, latency_ms=0.05)


wrote runs/r32/elevation/host-cpu/astar_classical__seed1234__2026-05-12T07-05-31Z
wrote runs/r32/elevation/host-cpu/greedy_classical__seed1234__2026-05-12T07-05-31Z


## 5. Leaderboard read-back

Pull the rows we just wrote and inspect the canonical columns.

In [5]:
lb = pe.load_leaderboard(RESULTS_ROOT)
display_cols = ['method_name','family','architecture','decoder_variant',
                'resolution','cost_model','seed','n_samples',
                'cost_regret_median','cost_regret_p90','path_assd',
                'path_hausdorff_p95','success_rate','latency_e2e_ms_p50',
                'package_version','timestamp_utc']
lb.df[display_cols].head()


,method_name,family,architecture,decoder_variant,resolution,cost_model,seed,n_samples,cost_regret_median,cost_regret_p90,path_assd,path_hausdorff_p95,success_rate,latency_e2e_ms_p50,package_version,timestamp_utc
0,astar_classical,classical,analytic,analytic_a_star,32,elevation,1234,8,0.000000,0.000000,0.000000,0.000000,1.00000,0.50,0.3.0,2026-05-10T16-19-19Z
1,greedy_classical,classical,analytic,manhattan_descent,32,elevation,1234,8,0.087332,0.123439,0.028163,0.054626,1.00000,0.05,0.3.0,2026-05-10T16-19-19Z
2,seg-convnextv2_atto_baseline,dense_segmentation,convnextv2_atto,BCE_threshold,256,elevation,1234,1024,0.036936,0.142076,0.110557,0.196772,0.99707,11.27,0.3.0,2026-05-10T16-19-22Z
3,seg-convnextv2_atto_baseline,dense_segmentation,convnextv2_atto,BCE_threshold,256,elevation,2026,1024,0.037426,0.142858,0.109595,0.194260,0.99707,11.27,0.3.0,2026-05-10T16-19-22Z
4,seg-convnextv2_atto_baseline,dense_segmentation,convnextv2_atto,BCE_threshold,256,elevation,3407,1024,0.040326,0.145074,0.109070,0.193913,0.99707,11.27,0.3.0,2026-05-10T16-19-22Z


## 6. One quick comparison figure

We use the bundled `plot_primary_bars` helper which reads the
leaderboard DataFrame directly. Bar colours encode the family
(classical / dense_regression / dense_segmentation / graph).

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pe.plot_primary_bars(lb.df, metric='cost_regret_median', ax=axes[0])
pe.plot_primary_bars(lb.df, metric='path_hausdorff_p95',   ax=axes[1])
fig.suptitle('Synthetic 32x32 leaderboard — A* (oracle) vs Greedy (weak)', fontsize=11)
fig.tight_layout()
plt.show()


## 7. Summary

This notebook illustrated the full path the protocol expects from
**every** PASSAGE submission:

1. Implement the `Planner` ABC.
2. Build a list of `Scene`s (elevation + obstacles + start/goal + oracle path).
3. Call `score_planner` to produce the six primary metrics.
4. Wrap the result in a `Submission` carrying every metadata column the
   leaderboard surfaces.
5. Call `pe.submit(...)`. The on-disk bundle (`metrics.json`,
   `sidecar.json`, optional `per_sample.npz`, optional
   `trustworthiness.json`) lands under `results/runs/<run_id>/`; the
   leaderboard CSV is appended atomically.

`02_score_released_baselines.ipynb` runs the same flow on the released
vision and graph NPZ files — no re-training. `03_leaderboard.ipynb`
aggregates and visualises everything in one place.